# Protograph pipeline runtime analysis

Stage-level wall-clock breakdown for the six protograph variants (`p{1,2,3}_classic`, `p{1,2,3}_bound`) on the DLCC synthetic test cases.

**Excluded** (not timed):
- protograph construction (`write_protographs`)
- protograph random-walk generation
- instance-graph random-walk generation

**Included stages** (per variant):

| stage | description |
|---|---|
| `pretrain_sgns` | skip-gram on protograph walks (5 epochs) |
| `normalize_codes` | L2 re-scale pretrained rows to `TARGET_NORM` |
| `concept_bound` | Hadamard-bound init vectors (`*_bound` only) |
| `build_vocab` | gensim vocab scan of instance-walk corpus |
| `init_vectors` | transfer protograph codes + MASCHInE / bound init |
| `finetune_sgns` | skip-gram on instance walks (5 epochs, LR 0.0025) |

`*_classic` and `*_bound` on the same protograph **share** pretrain; the tables below report pretrain once per kind and full per-variant totals.

Raw timings: `notebooks/runtime/results.json` (generated by `scripts/run_runtime.py`).

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
RESULTS_JSON = ROOT / "notebooks" / "runtime" / "results.json"
RUN_SCRIPT = ROOT / "scripts" / "run_runtime.py"

TCS = [f"tc{i:02d}" for i in range(1, 17) if i != 4]
PROTO_KINDS = ["p1", "p2", "p3"]
CLASSIC = [f"{k}_classic" for k in PROTO_KINDS]
BOUND = [f"{k}_bound" for k in PROTO_KINDS]
ALL_VARIANTS = CLASSIC + BOUND
STAGES = [
    "pretrain_sgns",
    "normalize_codes",
    "concept_bound",
    "build_vocab",
    "init_vectors",
    "finetune_sgns",
]
STAGE_LABELS = {
    "pretrain_sgns": "pretrain SGNS",
    "normalize_codes": "normalize codes",
    "concept_bound": "concept bound",
    "build_vocab": "build vocab",
    "init_vectors": "init vectors",
    "finetune_sgns": "finetune SGNS",
}
STAGE_COLORS = {
    "pretrain_sgns": "#4C72B0",
    "normalize_codes": "#55A868",
    "concept_bound": "#C44E52",
    "build_vocab": "#8172B3",
    "init_vectors": "#CCB974",
    "finetune_sgns": "#64B5CD",
}

if not RESULTS_JSON.is_file() or len(json.loads(RESULTS_JSON.read_text())) < len(TCS):
    print("Running benchmark (cached walks only, no walk generation)...", flush=True)
    subprocess.run([sys.executable, str(RUN_SCRIPT)], check=True)

raw = json.loads(RESULTS_JSON.read_text(encoding="utf-8"))
print(f"Loaded timings for {len(raw)} TCs: {', '.join(sorted(raw))}")

## Protograph pretrain time (shared across classic / bound)

SGNS pretrain + code normalization on P1 / P2 / P3 walks. This is identical for `<p>_classic` and `<p>_bound`.

In [ ]:
pretrain_rows = []
for tc, payload in sorted(raw.items()):
    for kind, stages in payload["pretrain_by_kind"].items():
        pretrain_rows.append({
            "tc": tc,
            "kind": kind,
            **{k: stages[k] for k in ("pretrain_sgns", "normalize_codes", "total_pretrain")},
        })

pretrain_df = pd.DataFrame(pretrain_rows)
pretrain_summary = (
    pretrain_df.groupby("kind")[["pretrain_sgns", "normalize_codes", "total_pretrain"]]
    .agg(["mean", "std", "min", "max"])
    .round(2)
)
display(pretrain_summary)

fig, ax = plt.subplots(figsize=(8, 4))
means = pretrain_df.groupby("kind")["total_pretrain"].mean().reindex(PROTO_KINDS)
ax.bar(means.index, means.values, color=["#4C72B0", "#55A868", "#C44E52"])
ax.set_ylabel("seconds (mean over TCs)")
ax.set_title("Protograph pretrain + normalize (excl. walk generation)")
for i, v in enumerate(means.values):
    ax.text(i, v + 0.05, f"{v:.2f}s", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

## Per-variant stage breakdown (mean over TCs)

Full pipeline time per variant, stacked by stage.

In [ ]:
variant_rows = []
for tc, payload in sorted(raw.items()):
    for row in payload["variants"]:
        rec = {"tc": tc, "variant": row["variant"], "proto_kind": row["proto_kind"]}
        rec.update(row["stages"])
        variant_rows.append(rec)

var_df = pd.DataFrame(variant_rows)
mean_stages = var_df.groupby("variant")[STAGES + ["total"]].mean().reindex(ALL_VARIANTS)
display(mean_stages.round(2))

fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(ALL_VARIANTS))
x = np.arange(len(ALL_VARIANTS))
for stage in STAGES:
    vals = mean_stages[stage].values
    ax.bar(x, vals, bottom=bottom, label=STAGE_LABELS[stage], color=STAGE_COLORS[stage])
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels(ALL_VARIANTS, rotation=30, ha="right")
ax.set_ylabel("seconds (mean over TCs)")
ax.set_title("Pipeline stage breakdown (walk generation excluded)")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=9)
for i, tot in enumerate(mean_stages["total"].values):
    ax.text(i, tot + 0.1, f"{tot:.1f}s", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## Classic vs bound overhead

For each protograph kind, the bound variant adds `concept_bound` init construction on top of the same pretrain. Finetune dominates both.

In [ ]:
cmp_rows = []
for kind in PROTO_KINDS:
    classic = mean_stages.loc[f"{kind}_classic"]
    bound = mean_stages.loc[f"{kind}_bound"]
    cmp_rows.append({
        "kind": kind,
        "classic_total_s": classic["total"],
        "bound_total_s": bound["total"],
        "bound_overhead_s": bound["total"] - classic["total"],
        "concept_bound_s": bound["concept_bound"],
        "init_delta_s": bound["init_vectors"] - classic["init_vectors"],
        "finetune_delta_s": bound["finetune_sgns"] - classic["finetune_sgns"],
    })
cmp_df = pd.DataFrame(cmp_rows).set_index("kind")
display(cmp_df.round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col, title in zip(
    axes,
    ["classic_total_s", "bound_total_s"],
    ["classic total", "bound total"],
):
    ax.bar(PROTO_KINDS, cmp_df[col], color="#4C72B0")
    ax.set_title(f"Mean {title} (s)")
    ax.set_ylabel("seconds")
plt.tight_layout()
plt.show()

## Pretrain share of total pipeline

Fraction of end-to-end (non-walk) time spent in protograph SGNS pretrain.

In [ ]:
share = mean_stages["pretrain_sgns"] / mean_stages["total"] * 100
share_df = pd.DataFrame({"pretrain_share_pct": share.round(1)})
display(share_df)

post_pretrain = mean_stages["total"] - mean_stages["pretrain_sgns"]
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(ALL_VARIANTS))
ax.bar(x, mean_stages["pretrain_sgns"], label="pretrain SGNS", color="#4C72B0")
ax.bar(x, post_pretrain, bottom=mean_stages["pretrain_sgns"],
       label="init + finetune (+ bound)", color="#64B5CD")
ax.set_xticks(x)
ax.set_xticklabels(ALL_VARIANTS, rotation=30, ha="right")
ax.set_ylabel("seconds (mean over TCs)")
ax.set_title("Pretrain vs rest of pipeline")
ax.legend()
plt.tight_layout()
plt.show()

## Per-TC heatmap (total seconds)

End-to-end variant time across test cases.

In [ ]:
pivot = var_df.pivot(index="tc", columns="variant", values="total").reindex(columns=ALL_VARIANTS)
display(pivot.round(1))

fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(pivot))))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(ALL_VARIANTS)))
ax.set_xticklabels(ALL_VARIANTS, rotation=30, ha="right")
ax.set_yticks(range(len(pivot)))
ax.set_yticklabels(pivot.index)
ax.set_title("Total pipeline time by TC and variant (s)")
plt.colorbar(im, ax=ax, label="seconds")
plt.tight_layout()
plt.show()

## Summary

Key takeaways printed below from the aggregated timings.

In [ ]:
cfg = next(iter(raw.values()))["config"]
pretrain_means = pretrain_df.groupby("kind")["total_pretrain"].mean()
print("Configuration:")
print(f"  dim={cfg['dim']}, pretrain_epochs={cfg['pretrain_epochs']}, finetune_epochs={cfg['epochs']}")
print(f"  proto_walks_per_entity={cfg['proto_walks_per_entity']}, instance_walks_per_entity={cfg['walks_per_entity']}, depth={cfg['depth']}")
print(f"  workers={cfg['workers']}, seed={cfg['seed']}")
print()
print("Mean protograph pretrain + normalize (seconds):")
for kind in PROTO_KINDS:
    print(f"  {kind}: {pretrain_means[kind]:.2f}s")
print()
print("Mean total pipeline time per variant (seconds, walk gen excluded):")
for v in ALL_VARIANTS:
    print(f"  {v:>11}: {mean_stages.loc[v, 'total']:.2f}s  "
          f"(pretrain {mean_stages.loc[v, 'pretrain_sgns']:.2f}s, "
          f"finetune {mean_stages.loc[v, 'finetune_sgns']:.2f}s)")
print()
fastest = mean_stages["total"].idxmin()
slowest = mean_stages["total"].idxmax()
print(f"Fastest variant (mean): {fastest} ({mean_stages.loc[fastest, 'total']:.2f}s)")
print(f"Slowest variant (mean): {slowest} ({mean_stages.loc[slowest, 'total']:.2f}s)")
print(f"Finetune dominates: {mean_stages['finetune_sgns'].mean() / mean_stages['total'].mean() * 100:.0f}% of mean total time")